In [ ]:
!pip install -q openai tqdm pyarrow pandas numpy

In [ ]:
# ======================================================================
# A. Install + Config (Colab)
# ======================================================================


from google.colab import drive, userdata
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os, re, json, time, hashlib
from datetime import datetime
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI

# ---- PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"
INPUT_FILE_DRIVE = f"{BASE}/preprocess/forum/reddit_2023_2025/reddit_clean_2023_2025.csv"
INPUT_FILE_LOCAL = "/mnt/data/reddit_clean_2023_2025.csv"
INPUT_FILE = INPUT_FILE_DRIVE if os.path.exists(INPUT_FILE_DRIVE) else INPUT_FILE_LOCAL

OUTPUT_FILE     = f"{BASE}/labeled/reddit/reddit_2023_2025_property_labeled.csv"
CHECKPOINT_FILE = f"{BASE}/labeled/checkpoints/reddit/reddit_labeling_checkpoint.csv"
HASH_PATH       = f"{BASE}/labeled/checkpoints/reddit/reddit_hashes.parquet"
AN_OUT          = f"{BASE}/analytics/reddit_2023_2025"

for p in [OUTPUT_FILE, CHECKPOINT_FILE, HASH_PATH]:
    os.makedirs(os.path.dirname(p), exist_ok=True)
os.makedirs(AN_OUT, exist_ok=True)

# ---- OpenAI Settings ----
OPENAI_MODEL     = "gpt-4o"     # or "gpt-5" if enabled
MAX_RETRIES      = 3
BATCH_SIZE       = 10
RATE_LIMIT_DELAY = 0.8          # seconds between calls

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_FILE:", OUTPUT_FILE)
print("CHECKPOINT_FILE:", CHECKPOINT_FILE)
print("HASH_PATH:", HASH_PATH)
print("AN_OUT:", AN_OUT)

# ======================================================================
# B. OpenAI client
# ======================================================================
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        raise RuntimeError("No OPENAI_API_KEY found in Colab → Runtime → Secrets.")
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    client = OpenAI(api_key=OPENAI_API_KEY)

    _ = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=4
    )
    print(f"✓ OpenAI client initialized — model: {OPENAI_MODEL}")
except Exception as e:
    print("✗ Could not initialize OpenAI:", e)
    raise





INPUT_FILE: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025/reddit_clean_2023_2025.csv
OUTPUT_FILE: /content/drive/MyDrive/PropInsight/labeled/reddit/reddit_2023_2025_property_labeled.csv
CHECKPOINT_FILE: /content/drive/MyDrive/PropInsight/labeled/checkpoints/reddit/reddit_labeling_checkpoint.csv
HASH_PATH: /content/drive/MyDrive/PropInsight/labeled/checkpoints/reddit/reddit_hashes.parquet
AN_OUT: /content/drive/MyDrive/PropInsight/analytics/reddit_2023_2025
✓ OpenAI client initialized — model: gpt-4o


In [ ]:
# ======================================================================
# C. Helper functions
# ======================================================================
_ALLOWED_SENT  = {"positive","neutral","negative"}
_ALLOWED_PRICE = {"rising","neutral","falling"}
_ALLOWED_EMO   = {"joy","anger","fear","trust","anticipation","surprise","sadness","disgust","neutral"}
_ALLOWED_ABSA  = {"positive","neutral","negative","none"}

def _safe_json_val(s):
    if isinstance(s, (list, dict)) or s is None:
        return s
    if isinstance(s, str):
        t = s.strip()
        if t.startswith("{") or t.startswith("["):
            try:
                return json.loads(t)
            except Exception:
                return None
    return None

def _first_n_true_flags(row: pd.Series, n=8) -> List[str]:
    hits = []
    for c in row.index:
        if c.startswith("rx_"):
            v = row.get(c, False)
            try:
                if bool(v): hits.append(c.replace("rx_",""))
            except Exception:
                pass
    return sorted(hits)[:n]

def _top_entity_labels(row: pd.Series, n=6) -> List[str]:
    ents = row.get("entities", [])
    if isinstance(ents, str):
        parsed = _safe_json_val(ents)
        if parsed is not None:
            ents = parsed
    labels = []
    if isinstance(ents, list):
        for e in ents:
            lab = None
            if isinstance(e, dict):
                lab = e.get("label") or e.get("type")
            elif isinstance(e, (list, tuple)) and len(e) >= 2:
                lab = e[1]
            if lab:
                labels.append(str(lab))
    uniq = list(dict.fromkeys(labels))
    return uniq[:n]

def _clamp_set(val, allowed, default):
    v = (str(val or "")).lower().strip()
    return v if v in allowed else default

def normalize_llm_output(d: Dict) -> Dict:
    d = dict(d or {})
    d["overall_sentiment"]       = _clamp_set(d.get("overall_sentiment"),       _ALLOWED_SENT,  "neutral")
    d["policy_sentiment"]        = _clamp_set(d.get("policy_sentiment"),        _ALLOWED_SENT,  "neutral")
    d["price_sentiment"]         = _clamp_set(d.get("price_sentiment"),         _ALLOWED_PRICE, "neutral")
    d["affordability_sentiment"] = _clamp_set(d.get("affordability_sentiment"), _ALLOWED_SENT,  "neutral")
    d["emotion"]                 = _clamp_set(d.get("emotion"),                 _ALLOWED_EMO,   "neutral")

    for k in ["location","PolicyMentioned","Aspect","Entity","CulturalContext"]:
        v = d.get(k, "none")
        v = "none" if (v is None or str(v).strip() == "") else str(v).strip()
        d[k] = v

    absa_in = d.get("absa") or {}
    if isinstance(absa_in, str):
        maybe = _safe_json_val(absa_in)
        absa_in = maybe if isinstance(maybe, dict) else {}
    norm_absa = {}
    for k in ["price","affordability","policy","location","investment","rental","lifestyle","supply","demand"]:
        norm_absa[k] = _clamp_set(absa_in.get(k), _ALLOWED_ABSA, "none")
    d["absa"] = norm_absa
    return d

def labels_are_valid(d: Dict) -> bool:
    try:
        return (
            d.get("overall_sentiment") in _ALLOWED_SENT and
            d.get("policy_sentiment")  in _ALLOWED_SENT and
            d.get("price_sentiment")   in _ALLOWED_PRICE and
            d.get("affordability_sentiment") in _ALLOWED_SENT and
            d.get("emotion") in _ALLOWED_EMO
        )
    except Exception:
        return False

def row_is_labeled_ok(row: pd.Series) -> bool:
    d = {
        "overall_sentiment":       (row.get("overall_sentiment") or "").strip().lower(),
        "policy_sentiment":        (row.get("policy_sentiment") or "").strip().lower(),
        "price_sentiment":         (row.get("price_sentiment") or "").strip().lower(),
        "affordability_sentiment": (row.get("affordability_sentiment") or "").strip().lower(),
        "emotion":                 (row.get("emotion") or "").strip().lower(),
    }
    return labels_are_valid(d)

def create_llm_prompt_reddit(body: str,
                             subreddit: str,
                             has_singlish: bool,
                             rx_flags: List[str],
                             entity_labels: List[str]) -> str:
    rx_str  = ", ".join(rx_flags) if rx_flags else "None"
    ent_str = ", ".join(entity_labels) if entity_labels else "None"
    subr    = subreddit or "unknown"

    return f"""Analyze this Singapore Reddit post/comment about housing/property and extract policy-aware sentiment and context.

SUBREDDIT: {subr}
Singlish detected: {has_singlish}
Regex-topic flags: {rx_str}
Domain entity labels: {ent_str}

TEXT (truncated if long):
{body[:3000]}

Return ONLY valid JSON with exactly these keys:
{{
  "overall_sentiment": "positive|neutral|negative",
  "policy_sentiment": "positive|neutral|negative",
  "price_sentiment": "rising|neutral|falling",
  "affordability_sentiment": "positive|neutral|negative",
  "PolicyMentioned": "comma-separated terms or 'none'",
  "location": "estate/district/MRT or 'none'",
  "emotion": "joy|anger|fear|trust|anticipation|surprise|sadness|disgust|neutral",
  "Aspect": "price|affordability|policy|location|lifestyle|investment|rental|supply|demand|none",
  "Entity": "key named thing (estate, project, MRT, developer) or 'none'",
  "CulturalContext": "short phrase if any Singlish/cultural cue else 'none'",
  "absa": {{
      "price": "positive|neutral|negative|none",
      "affordability": "positive|neutral|negative|none",
      "policy": "positive|neutral|negative|none",
      "location": "positive|neutral|negative|none",
      "investment": "positive|neutral|negative|none",
      "rental": "positive|neutral|negative|none",
      "lifestyle": "positive|neutral|negative|none",
      "supply": "positive|neutral|negative|none",
      "demand": "positive|neutral|negative|none"
  }}
}}"""

def call_openai_api(client: OpenAI, prompt: str) -> Dict:
    delay = 1.0
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role":"system","content":"You are an expert on Singapore housing discourse (Reddit). Output ONLY valid JSON."},
                    {"role":"user","content": prompt}
                ],
                temperature=0.2,
                max_tokens=600,
                response_format={"type":"json_object"},
            )
            data = json.loads(resp.choices[0].message.content)
            data = normalize_llm_output(data)
            return data
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f"✗ OpenAI error (final): {e}")
                break
            sleep = min(delay * 1.8, 20.0) + float(np.random.uniform(0, 0.8))
            print(f"⚠ OpenAI error: {e}. Retry in {sleep:.1f}s")
            time.sleep(sleep)
            delay = sleep

    return normalize_llm_output({
        "overall_sentiment":"neutral",
        "policy_sentiment":"neutral",
        "price_sentiment":"neutral",
        "affordability_sentiment":"neutral",
        "PolicyMentioned":"none",
        "location":"none",
        "emotion":"neutral",
        "Aspect":"none",
        "Entity":"none",
        "CulturalContext":"none",
        "absa":{
            "price":"none","affordability":"none","policy":"none","location":"none",
            "investment":"none","rental":"none","lifestyle":"none","supply":"none","demand":"none"
        }
    })

def process_one_row(row: pd.Series, client: OpenAI) -> Dict:
    body       = str(row.get("body") or row.get("clean_text") or "")
    subreddit  = str(row.get("subreddit") or "")
    has_sing   = bool(row.get("has_singlish", False))
    rx_hits    = _first_n_true_flags(row, n=8)
    ent_labels = _top_entity_labels(row, n=6)
    prompt = create_llm_prompt_reddit(body, subreddit, has_sing, rx_hits, ent_labels)
    out = call_openai_api(client, prompt)
    out["subreddit"] = subreddit
    return out

def _body_hash(s: str) -> str:
    s = (s or "").strip().lower()
    return hashlib.md5(s.encode("utf-8")).hexdigest()


In [ ]:
# ======================================================================
# D. Load + resume safely
# ======================================================================
print("Loading reddit data…")
df = pd.read_csv(INPUT_FILE)
print(f"✓ Loaded {len(df)} rows")
print("Columns (first 24):", df.columns.tolist()[:24])

if "body" not in df.columns:
    title_col   = next((c for c in ["title","post_title"] if c in df.columns), None)
    self_col    = next((c for c in ["selftext","post_text","self_text"] if c in df.columns), None)
    comment_col = next((c for c in ["comment_body","comment_text","text"] if c in df.columns), None)
    title_series   = df[title_col].astype(str)   if title_col   else pd.Series([""]*len(df), index=df.index)
    self_series    = df[self_col].astype(str)    if self_col    else pd.Series([""]*len(df), index=df.index)
    comment_series = df[comment_col].astype(str) if comment_col else pd.Series([""]*len(df), index=df.index)
    df["body"] = (title_series + "\n\n" + self_series + "\n\n" + comment_series).str.strip()

if "hash" not in df.columns:
    df["hash"] = df["body"].map(_body_hash)

pre = len(df)
df = df[df["body"].astype(str).str.len() >= 60].copy()
df.reset_index(drop=True, inplace=True)
print(f"[Prefilter] removed {pre - len(df)} short rows; remain {len(df)}")

_date_cols = [c for c in ["created_utc", "created_ts", "timestamp", "date", "datetime"] if c in df.columns]
if _date_cols:
    df["Datetime"] = pd.to_datetime(df[_date_cols[0]], errors="coerce", utc=True).dt.tz_convert("Asia/Singapore")
else:
    df["Datetime"] = pd.Timestamp.utcnow().tz_localize("UTC").tz_convert("Asia/Singapore")

df["Source"] = "Reddit"
df["SinglishDetected"] = df.get("has_singlish", False).astype(bool)

_up = df.get("upvotes") if "upvotes" in df.columns else df.get("score")
_re = df.get("replies") if "replies" in df.columns else df.get("num_comments")
df["EngagementScore"] = ( (_up.fillna(0).astype(float) if _up is not None else 0)
                        + (_re.fillna(0).astype(float) if _re is not None else 0) )

for _col in ["Aspect","Entity","CulturalContext"]:
    if _col not in df.columns:
        df[_col] = None

if "policy_mentioned" in df.columns and "PolicyMentioned" not in df.columns:
    df.rename(columns={"policy_mentioned": "PolicyMentioned"}, inplace=True)
elif "PolicyMentioned" not in df.columns:
    df["PolicyMentioned"] = None

try:
    seen = pd.read_parquet(HASH_PATH)
    _seen_set = set(seen["hash"].astype(str))
except Exception:
    _seen_set = set()

if os.path.exists(CHECKPOINT_FILE):
    try:
        ckp = pd.read_csv(CHECKPOINT_FILE)
        if "hash" in ckp.columns:
            before_cols = set(df.columns)
            df = df.set_index("hash").combine_first(ckp.set_index("hash")).reset_index()
            for col in set(df.columns)-before_cols:
                df[col] = df[col]
            print("✓ Merged checkpoint by hash")
        else:
            print("⚠ Checkpoint missing 'hash' column; skipping merge.")
    except Exception as e:
        print("⚠ Could not merge checkpoint:", e)

label_columns = [
    'overall_sentiment','price_sentiment','policy_sentiment','affordability_sentiment',
    'location','PolicyMentioned','emotion','subreddit','Aspect','Entity','CulturalContext','absa'
]
for col in label_columns:
    if col not in df.columns:
        df[col] = None

mask_valid_now = df.apply(row_is_labeled_ok, axis=1)
new_to_label = (~df["hash"].isin(_seen_set) & ~mask_valid_now).sum()
print(f"New bodies to label: {new_to_label} / {len(df)}")



Loading reddit data…
✓ Loaded 3275 rows
Columns (first 24): ['hash', 'date', 'year', 'quarter', 'month', 'subreddit', 'post_id', 'comment_id', 'author', 'comment_author', 'title', 'selftext', 'comment_body', 'body', 'url', 'permalink', 'score_post', 'comment_score', 'num_comments', 'has_singlish', 'singlish_terms', 'entities']
[Prefilter] removed 0 short rows; remain 3275
New bodies to label: 3275 / 3275


In [ ]:
# ======================================================================
# E. Main labeling loop
# ======================================================================
print("="*70); print("Starting labeling…"); print("="*70, "\n")

total_rows = len(df)
pbar = tqdm(total=int((~mask_valid_now).sum()), desc="Labeling", unit="items")
processed_count, errors = 0, []

try:
    for idx in range(total_rows):
        if row_is_labeled_ok(df.iloc[idx]) or (df.at[idx, 'hash'] in _seen_set):
            pbar.update(1 if not mask_valid_now.iloc[idx] else 0)
            continue

        try:
            labels = process_one_row(df.iloc[idx], client)
            labels = normalize_llm_output(labels)

            for k, v in labels.items():
                df.at[idx, k] = v

            if row_is_labeled_ok(df.iloc[idx]):
                processed_count += 1

            pbar.update(1)
            mask_valid_now.iloc[idx] = row_is_labeled_ok(df.iloc[idx])

            if processed_count % BATCH_SIZE == 0:
                df.to_csv(CHECKPOINT_FILE, index=False)

                ok_mask = df.apply(row_is_labeled_ok, axis=1)
                to_add = df.loc[ok_mask, ['hash']].drop_duplicates()
                if len(to_add):
                    try:
                        cur = pd.read_parquet(HASH_PATH)
                    except Exception:
                        cur = pd.DataFrame(columns=["hash"])
                    merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
                    merged.to_parquet(HASH_PATH, index=False)
                    _seen_set.update(set(to_add['hash']))

            time.sleep(RATE_LIMIT_DELAY)

        except KeyboardInterrupt:
            print("\nStopping… saving checkpoint.")
            df.to_csv(CHECKPOINT_FILE, index=False)
            raise

        except Exception as e:
            msg = f"Row {idx}: {e}"
            errors.append(msg)
            print("⚠", msg)
            continue

    pbar.close()
except KeyboardInterrupt:
    pbar.close()

# Final hash save (only valid rows)
try:
    cur = pd.read_parquet(HASH_PATH)
except Exception:
    cur = pd.DataFrame(columns=["hash"])
ok_mask = df.apply(row_is_labeled_ok, axis=1)
to_add = df.loc[ok_mask, ['hash']].drop_duplicates()
merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
merged.to_parquet(HASH_PATH, index=False)
_seen_set.update(set(to_add['hash']))

print("\nDone. Processed:", processed_count, "| Errors:", len(errors))
if errors:
    print("\nError summary (first 5):")
    for e in errors[:5]:
        print(" -", e)


Starting labeling…



Labeling:   0%|          | 0/3275 [00:00<?, ?items/s]


Done. Processed: 3275 | Errors: 0


In [ ]:
# ======================================================================
# F. Post-label normalization & Analytics
# ======================================================================
rename_map = {
    "overall_sentiment": "Sentiment",
    "policy_sentiment": "PolicySentiment",
    "price_sentiment": "PriceSentiment",
    "affordability_sentiment": "AffordabilitySentiment",
    "location": "Location",
    "emotion": "Emotion"
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

need_ctx = (df["CulturalContext"].fillna("none").str.lower().eq("none")) & df["SinglishDetected"]
if "body" in df.columns:
    patt = r"(lah|leh|lor|liao|sia|aiyah|walau|steady|shiok|kanchiong|pai seh|bo[ -]?jio|mai hiam bei pai)"
    ctx = df.loc[need_ctx, "body"].str.lower().str.extract(patt, expand=False)
    df.loc[need_ctx, "CulturalContext"] = np.where(ctx.notna(), ctx.fillna("").str.strip(), "singlish (general)")

df["SinglishDetected"] = df["SinglishDetected"].fillna(False).astype(bool)

def _to_json(x):
    try:
        if isinstance(x, str):
            json.loads(x)
            return x
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps({}, ensure_ascii=False)

df["ABSA"] = df["absa"].apply(_to_json) if "absa" in df.columns else json.dumps({})

try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Saved labeled dataset → {OUTPUT_FILE} | rows={len(df)}")
except Exception as e:
    print("✗ Error saving:", e)
    print("Checkpoint at:", CHECKPOINT_FILE)

def _vc(col):
    if col in df.columns:
        print(df[col].value_counts(dropna=True))
    else:
        print(f"(missing: {col})")

print("\n=== SUMMARY ===")
print("\n📊 Overall Sentiment:");      _vc("Sentiment")
print("\n🏛 Policy Sentiment:");       _vc("PolicySentiment")
print("\n📈 Price Sentiment:");        _vc("PriceSentiment")
print("\n💰 Affordability Sentiment:");_vc("AffordabilitySentiment")
print("\n😊 Emotion:");                _vc("Emotion")

print("\nTop policy terms (model output):")
if "PolicyMentioned" in df.columns:
    pm = df[df["PolicyMentioned"].notna() & (df["PolicyMentioned"].str.lower()!="none")]["PolicyMentioned"]
    if len(pm):
        print(pm.str.split(",").explode().str.strip().str.lower().value_counts().head(20))
    else:
        print("(none)")
else:
    print("(missing)")

print("\nSubreddit distribution:"); _vc("subreddit")

# -------- Analytics tables --------
tmp = df.copy()
_map = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
tmp["SentimentScore"] = tmp["Sentiment"].map(_map).fillna(0.0)
tmp["week"] = tmp["Datetime"].dt.to_period("W").dt.start_time

sent_idx_overall = tmp.groupby("week", dropna=False)["SentimentScore"].mean().reset_index().rename(columns={"SentimentScore":"SentimentIndex"})
sent_idx_loc     = tmp.groupby(["week","Location"], dropna=False)["SentimentScore"].mean().reset_index().rename(columns={"SentimentScore":"SentimentIndex"})
sent_idx_overall.to_csv(f"{AN_OUT}/sentiment_index_overall_weekly.csv", index=False)
sent_idx_loc.to_csv(f"{AN_OUT}/sentiment_index_by_location_weekly.csv", index=False)

tsm = pd.pivot_table(tmp, index="Aspect", columns="Sentiment",
                     values="hash" if "hash" in tmp.columns else "body",
                     aggfunc="count", fill_value=0, dropna=False)
tsm.to_csv(f"{AN_OUT}/topic_sentiment_matrix.csv")

if "Source" in tmp.columns:
    cc = pd.pivot_table(tmp, index=["Source","Aspect"], columns="Sentiment",
                        values="Location", aggfunc="count",
                        fill_value=0, dropna=False)
    cc.to_csv(f"{AN_OUT}/community_comparison.csv")

tmp["EWS"] = tmp["SentimentScore"] * tmp["EngagementScore"].clip(lower=0)
ews = tmp.groupby("week", dropna=False)["EWS"].sum().reset_index()
ews.to_csv(f"{AN_OUT}/engagement_weighted_sentiment_weekly.csv", index=False)

try:
    absa_df = pd.json_normalize(tmp["ABSA"].apply(json.loads)).fillna("none")
    absa_df["week"] = tmp["week"].values
    absa_df["Location"] = tmp["Location"].values
    absa_counts = {}
    for aspect in ["price","affordability","policy","location","investment","rental","lifestyle","supply","demand"]:
        pos = (absa_df[aspect].str.lower() == "positive").groupby(absa_df["week"]).sum().rename(f"{aspect}_positive")
        neu = (absa_df[aspect].str.lower() == "neutral").groupby(absa_df["week"]).sum().rename(f"{aspect}_neutral")
        neg = (absa_df[aspect].str.lower() == "negative").groupby(absa_df["week"]).sum().rename(f"{aspect}_negative")
        absa_counts[aspect] = pd.concat([pos, neu, neg], axis=1)
    absa_weekly = None
    for aspect, table in absa_counts.items():
        absa_weekly = table if absa_weekly is None else absa_weekly.join(table, how="outer")
    absa_weekly = absa_weekly.fillna(0).reset_index()
    absa_weekly.to_csv(f"{AN_OUT}/absa_weekly_counts.csv", index=False)
except Exception as e:
    print("⚠ ABSA analytics flattening error:", e)

print("✓ Analytics tables saved in:", AN_OUT)


✓ Saved labeled dataset → /content/drive/MyDrive/PropInsight/labeled/reddit/reddit_2023_2025_property_labeled.csv | rows=3275

=== SUMMARY ===

📊 Overall Sentiment:
Sentiment
neutral     2203
negative     758
positive     314
Name: count, dtype: int64

🏛 Policy Sentiment:
PolicySentiment
neutral     2434
negative     586
positive     255
Name: count, dtype: int64

📈 Price Sentiment:
PriceSentiment
neutral    2277
rising      903
falling      95
Name: count, dtype: int64

💰 Affordability Sentiment:
AffordabilitySentiment
neutral     2061
negative     914
positive     300
Name: count, dtype: int64

😊 Emotion:
Emotion
anticipation    1291
neutral         1042
fear             231
disgust          179
anger            162
sadness          138
trust            112
surprise          75
joy               45
Name: count, dtype: int64

Top policy terms (model output):
PolicyMentioned
bto                 308
cpf                 239
hdb                 136
mop                  95
absd            

/tmp/ipython-input-2416540276.py:69: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  tmp["week"] = tmp["Datetime"].dt.to_period("W").dt.start_time
